In [0]:
df = spark.table("clean_transactions_dlt")
df.show(5)

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [c for c in df.columns if c not in ["Class"]]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
model_data = assembler.transform(df)

model_data.select("features", "Class").show(5)

In [0]:
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count()}")
print(f"Test rows: {test_data.count()}")

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="Class")
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)
predictions.select("Class", "prediction", "probability").show(10)


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="Class", predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel"})

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")

In [0]:
precision_fraud = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
recall_fraud = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})

print(f"Fraud Precision: {precision_fraud}")
print(f"Fraud Recall: {recall_fraud}")